# 🌾 RICE DATASET EXTRACTION PIPELINE

This notebook runs the standalone `DATASET_BUILDER/src/rice_dataset` pipeline.
Raw inputs must be one workbook row and one image per `M####` ID, with images directly in `1_Raw_Images`.

In [1]:
# [Section 1]: Mount Google Drive and locate PROJECT_ROOT
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

override = os.environ.get("RICAI_PROJECT_ROOT")
candidates = [Path(override)] if override else [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES"),
]
shortcut_root = Path("/content/drive/.shortcut-targets-by-id")
if not override and shortcut_root.exists():
    candidates.extend(sorted(shortcut_root.glob(
        "*/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES"
    )))
PROJECT_ROOT = next((
    candidate for candidate in candidates
    if (candidate / "DATASET_BUILDER/config/extraction_config.json").is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate MAIN_SOURCES. Set RICAI_PROJECT_ROOT to the absolute project path."
    )
print(f"📂 PROJECT_ROOT: {PROJECT_ROOT}")

Mounted at /content/drive
📂 PROJECT_ROOT: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES


In [2]:
# [Section 2 & 3]: Add package path and install runtime dependencies
import sys

src_dir = PROJECT_ROOT / "DATASET_BUILDER" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

!pip install -q ultralytics sahi tensorflow openpyxl pandas numpy opencv-python

import torch
print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.0 MB/s eta 0:00:00
⚡ PyTorch Version: 2.11.0+cu128
🎮 CUDA Available: True
   Device: Tesla T4


In [3]:
# [Section 4]: Load and Display Validated Extraction Config
from rice_dataset.config import ExtractionConfig

config_file = PROJECT_ROOT / "DATASET_BUILDER" / "config" / "extraction_config.json"
config = ExtractionConfig.from_file(config_file, project_root_override=PROJECT_ROOT)

print("📋 Loaded Extraction Configuration:")
print(f"   - Contract Version  : {config.contract_version}")
print(f"   - SAHI Slice Size   : {config.sahi_slice_size}")
print(f"   - SAHI Overlap      : {config.sahi_overlap_ratio}")
print(f"   - CNN Whole Thresh  : {config.cnn_whole_confidence}")
print(f"   - Physical Packing  : {config.physical_packing_fraction}")
print(f"   - Hybrid Packing    : {config.trained_hybrid_packing_fraction}")
print(f"   - Thickness Ratio   : {config.whole_grain_thickness_ratio}")

📋 Loaded Extraction Configuration:
   - Contract Version  : code-main-2026-09-22
   - SAHI Slice Size   : 640
   - SAHI Overlap      : 0.25
   - CNN Whole Thresh  : 0.9
   - Physical Packing  : 0.55
   - Hybrid Packing    : 0.62
   - Thickness Ratio   : 0.8


In [4]:
# [Section 5]: Preflight Model Files & Paths
yolo_path = config.get_resolved_yolo_model_path()
cnn_path = config.get_resolved_cnn_model_path()
workbook_path = config.get_resolved_workbook_path()
images_dir = config.get_resolved_raw_images_dir()

print("🔍 Verifying Paths:")
print(f"   - YOLO Model : {'✅' if yolo_path.exists() else '❌'} {yolo_path}")
print(f"   - CNN Model  : {'✅' if cnn_path.exists() else '❌'} {cnn_path}")
print(f"   - Workbook   : {'✅' if workbook_path.exists() else '❌'} {workbook_path}")
print(f"   - Images Dir : {'✅' if images_dir.exists() else '❌'} {images_dir}")

assert yolo_path.exists(), f"Missing YOLO model at {yolo_path}"
assert cnn_path.exists(), f"Missing CNN model at {cnn_path}"
assert workbook_path.exists(), f"Missing Workbook at {workbook_path}"
assert images_dir.exists(), f"Missing Images Dir at {images_dir}"

🔍 Verifying Paths:
   - YOLO Model : ✅ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/all-new-data-v1.yolov8_yolov8s-seg_trained/weights/best.pt
   - CNN Model  : ✅ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras
   - Workbook   : ✅ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/DATASET_BUILDER/2_Manual_Records/manual_data.xlsx
   - Images Dir : ✅ /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/DATASET_BUILDER/1_Raw_Images


In [5]:
# [Section 6 & 7]: Validate canonical workbook and flat raw images
from rice_dataset.io.manual_records import load_manual_records
from rice_dataset.io.image_index import validate_flat_inventory

records, invalid_rows = load_manual_records(workbook_path)
if invalid_rows:
    preview = [f"{row['sample_id']}: {row['error']}" for row in invalid_rows[:5]]
    raise ValueError(f"{len(invalid_rows)} invalid workbook row(s): {preview}")

inventory = validate_flat_inventory(
    records,
    images_dir,
    prefix=config.id_prefix,
    digits=config.id_digits,
    require_exact_match=True,
)
print(
    f"✅ Input inventory: {inventory['record_count']} workbook rows, "
    f"{inventory['image_count']} flat images, "
    f"{inventory['unique_record_count']} unique Sample_ID values."
)

✅ Input inventory: 261 workbook rows, 261 flat images, 261 unique Sample_ID values.


In [6]:
# [Section 8]: Initialize Models (Once)
from rice_dataset.pipeline import DatasetExtractionPipeline

pipeline = DatasetExtractionPipeline(config)
pipeline.initialize_models()
print("✅ All deep learning models initialized on device.")

📦 Loading SAHI YOLO model from: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/all-new-data-v1.yolov8_yolov8s-seg_trained/weights/best.pt...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ SAHI YOLO model loaded successfully.
📦 Loading DenseNet121 CNN model from: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras...
📦 Đang nạp mô hình CNN từ: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras...
✅ Nạp mô hình CNN thành công!
✅ DenseNet121 CNN model loaded successfully.
✅ All deep learning models initialized on device.


In [7]:
# [Section 9 & 10]: Run Batch Extraction & Show Failure Summary
# Set limit=None to process all rows, or limit=5 for a quick test run
results = pipeline.run_batch(limit=None, export=True)

import pandas as pd
df_res = pd.DataFrame(results)

print("\n📊 QC Status Breakdown:")
display(df_res["QC_Status"].value_counts())

failed_rows = df_res[df_res["QC_Status"] != "PASS"]
if not failed_rows.empty:
    print("\n⚠️ Non-PASS Samples Summary:")
    display(failed_rows[["Sample_ID", "Image_Status", "QC_Status", "QC_Reason"]].head(20))

🚀 Starting batch extraction for 261 sample records...
Performing prediction on 63 slices.
[001/261] M0001 -> PASS [20.71s]
Performing prediction on 63 slices.
[002/261] M0002 -> PASS [9.98s]
Performing prediction on 63 slices.
[003/261] M0003 -> PASS [26.16s]
Performing prediction on 63 slices.
[004/261] M0004 -> PASS [5.28s]
Performing prediction on 63 slices.
[005/261] M0005 -> PASS [6.64s]
Performing prediction on 63 slices.
[006/261] M0006 -> PASS [5.38s]
Performing prediction on 63 slices.
[007/261] M0007 -> PASS [7.22s]
Performing prediction on 63 slices.
[008/261] M0008 -> PASS [5.39s]
Performing prediction on 63 slices.
[009/261] M0009 -> PASS [5.07s]
Performing prediction on 63 slices.
[010/261] M0010 -> PASS [7.27s]
Performing prediction on 221 slices.
[011/261] M0011 -> PASS [18.52s]
Performing prediction on 221 slices.
[012/261] M0012 -> PASS [15.87s]
Performing prediction on 221 slices.
[013/261] M0013 -> PASS [19.61s]
Performing prediction on 221 slices.
[014/261] M0014 -

,count
QC_Status,
PASS,257
CONTAINER_DETECTION_FAILED,3
NO_CNN_WHOLE_GRAINS,1



⚠️ Non-PASS Samples Summary:


,Sample_ID,Image_Status,QC_Status,QC_Reason
74,M0075,FOUND,NO_CNN_WHOLE_GRAINS,CNN detected 0 whole grains at confidence thre...
203,M0204,FOUND,CONTAINER_DETECTION_FAILED,Container detection error: Tỉ lệ mép trong/mép...
230,M0231,FOUND,CONTAINER_DETECTION_FAILED,Container detection error: Không có vòng nào v...
259,M0260,FOUND,CONTAINER_DETECTION_FAILED,Container detection error: Không có vòng nào v...


In [8]:
# [Section 11]: Verify canonical export datasets
from rice_dataset.contracts import CANONICAL_COLUMNS, QCStatus

audit_p = config.get_resolved_audit_output_path()
final_csv_p = config.get_resolved_final_csv_path()
final_xlsx_p = config.get_resolved_final_xlsx_path()

df_audit = pd.read_csv(audit_p)
df_final = pd.read_csv(final_csv_p)
df_final_xlsx = pd.read_excel(final_xlsx_p)

for label, frame in (
    ("audit CSV", df_audit),
    ("training CSV", df_final),
    ("training XLSX", df_final_xlsx),
):
    assert frame.columns.tolist() == CANONICAL_COLUMNS, (
        f"{label}: export columns differ from the {len(CANONICAL_COLUMNS)}-column contract"
    )

pass_count = sum(row["QC_Status"] == QCStatus.PASS for row in results)
assert len(df_audit) == len(results), "Audit row count differs from batch results"
assert len(df_final) == pass_count, "Training CSV row count differs from PASS results"
assert len(df_final_xlsx) == pass_count, "Training XLSX row count differs from PASS results"
print(
    f"✅ Export contract: {len(CANONICAL_COLUMNS)} columns; "
    f"{len(df_audit)} audit rows; {pass_count} training rows."
)

✅ Export contract: 47 columns; 261 audit rows; 257 training rows.


In [9]:
# [Section 12]: Static and contract parity check
import subprocess

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "DATASET_BUILDER/scripts/verify_pipeline_parity.py"),
        "--static",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/DATASET_BUILDER/scripts/verify_pipeline_parity.py', '--static'], returncode=0)